# TCGA-BRCA RNA-seq external validation — GDC workflow

This notebook evaluates the frozen NIBFS top-20 panel in participant-matched TCGA-BRCA RNA-seq samples. It uses the repository-local helper `src/tcga_brca_rnaseq_external_validation.py` and the latest completed core run under `runs/`.

The GDC query uses `files.analysis.workflow_type`, does not require `files.is_latest`, links returned STAR-Counts files locally to selected tumor-normal pairs, and resolves duplicate file versions deterministically by update time.

Run `notebooks/01_main_NIBFS_core.ipynb` first, then run this notebook from top to bottom.


In [ ]:
from pathlib import Path
import json, re, shutil, subprocess, sys, zipfile
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)
markers = list(Path('/content/drive/MyDrive').rglob('NIBFS_REPRODUCIBILITY_PACKAGE.marker'))
if len(markers) != 1:
    raise RuntimeError(f'Expected exactly one repository marker, found {len(markers)}: {markers}')
PACKAGE_DIR = markers[0].parent.resolve()
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))

OUT = PACKAGE_DIR / 'results' / 'TCGA_BRCA_RNASEQ_EXTERNAL_VALIDATION'
BACKUP = PACKAGE_DIR / 'results' / 'archives'
OUT.mkdir(parents=True, exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightgbm', 'statsmodels', 'requests', 'pyyaml'
])

from src import tcga_brca_rnaseq_external_validation as rna

# Resolve the latest complete core run generated from this repository.
bundles = []
for matrix in list((PACKAGE_DIR / 'runs').rglob('harmonized_expression_matrix.csv.gz')) + list((PACKAGE_DIR / 'runs').rglob('harmonized_expression_matrix.csv')):
    table_dir = matrix.parent
    split_path = next((p for p in [
        table_dir / 'train_test_split_assignments.csv',
        table_dir / 'discovery_train_test_assignments.csv',
    ] if p.is_file()), None)
    panel_path = table_dir / 'final_NIBFS_gene_panel_k20.csv'
    if split_path is not None and panel_path.is_file():
        bundles.append((matrix.stat().st_mtime, matrix, split_path, panel_path))
if not bundles:
    raise FileNotFoundError(
        'No completed core run with harmonized matrix, split, and frozen panel was found. '
        'Run notebooks/01_main_NIBFS_core.ipynb first.'
    )
_, MATRIX, SPLIT, PANEL = sorted(bundles, reverse=True)[0]
print('Repository:', PACKAGE_DIR)
print('Matrix:', MATRIX)
print('Split :', SPLIT)
print('Panel :', PANEL)
print('Output:', OUT)

def fcol(df, names):
    lookup = {
        re.sub(r'[^a-z0-9]+', '', str(c).casefold()): str(c)
        for c in df.columns
    }
    for name in names:
        key = re.sub(r'[^a-z0-9]+', '', name.casefold())
        if key in lookup:
            return lookup[key]
    return None

def labels01(values):
    s = pd.Series(values)
    numeric = pd.to_numeric(s, errors='coerce')
    if numeric.notna().all() and set(numeric.astype(int).unique()).issubset({0, 1}):
        return numeric.astype(int).to_numpy()
    text = s.astype(str).str.strip().str.casefold()
    out = np.full(len(text), np.nan)
    out[text.str.contains(r'cancer|tumou?r|malignan|carcinoma', regex=True)] = 1
    out[text.str.contains(r'normal|control|healthy', regex=True)] = 0
    if np.isnan(out).any():
        raise ValueError('Label tidak dikenali: ' + str(sorted(s[np.isnan(out)].unique())))
    return out.astype(int)

expr = pd.read_csv(MATRIX)
sample_col = fcol(expr, ['GSM_ID', 'Sample_ID', 'sample']) or str(expr.columns[0])
expr[sample_col] = expr[sample_col].astype(str).str.strip()
expr = expr.set_index(sample_col)
expr.columns = expr.columns.astype(str).str.strip().str.upper()
expr = expr.loc[:, ~expr.columns.duplicated()]

split = pd.read_csv(SPLIT)
gsm_col = fcol(split, ['GSM_ID', 'Sample_ID', 'sample'])
set_col = fcol(split, ['Set', 'Subset', 'Role', 'Split'])
label_col = fcol(split, ['Label_binary', 'y', 'Label', 'Class'])
if gsm_col is None or set_col is None or label_col is None:
    raise KeyError('Kolom split tidak lengkap: ' + str(list(split.columns)))
mask = split[set_col].astype(str).str.casefold().str.contains(
    r'model-development|model development|development|train', regex=True
)
dev = split.loc[mask].copy()
dev[gsm_col] = dev[gsm_col].astype(str).str.strip()
dev = dev.drop_duplicates(gsm_col).set_index(gsm_col)
if len(dev) != 608:
    raise RuntimeError(f'Development harus 608, ditemukan {len(dev)}')
missing = dev.index.difference(expr.index)
if len(missing):
    raise RuntimeError(f'{len(missing)} development sample tidak ada di matrix')
Xdev = expr.loc[dev.index].copy()
ydev = pd.Series(labels01(dev[label_col]), index=Xdev.index)
if int(ydev.sum()) != 309 or int((1 - ydev).sum()) != 299:
    raise RuntimeError('Komposisi development bukan 309/299')
print('Development:', Xdev.shape, 'cancer/normal=', int(ydev.sum()), int((1-ydev).sum()))

panel_raw = pd.read_csv(PANEL)
gene_col = fcol(panel_raw, ['Gene', 'gene_symbol', 'Symbol'])
rank_col = fcol(panel_raw, ['Rank_NIBFS', 'Selection_rank', 'Rank'])
logfc_col = fcol(panel_raw, [
    'Full_training_logFC', 'logFC', 'log2FC', 'Training_log2FC'
])
if gene_col is None:
    raise KeyError('Kolom Gene tidak ditemukan pada panel')
panel_raw[gene_col] = panel_raw[gene_col].astype(str).str.strip().str.upper()
if rank_col:
    panel_raw = panel_raw.sort_values(rank_col)
genes = panel_raw[gene_col].drop_duplicates().tolist()[:20]
if genes != rna.EXPECTED_TOP20:
    raise RuntimeError('Panel archive berbeda dari frozen top-20:\n' + str(genes))
panel = pd.DataFrame({'Rank_NIBFS': range(1, 21), 'Gene': genes})
if logfc_col:
    z = pd.to_numeric(
        panel_raw.drop_duplicates(gene_col).set_index(gene_col)[logfc_col],
        errors='coerce'
    ).reindex(genes)
    panel['Discovery_logFC'] = z.to_numpy()
else:
    panel['Discovery_logFC'] = [rna.EXPECTED_LOGFC[g] for g in genes]
if panel['Discovery_logFC'].isna().any():
    panel['Discovery_logFC'] = [rna.EXPECTED_LOGFC[g] for g in genes]
panel['Discovery_direction'] = np.where(
    panel['Discovery_logFC'] > 0, 'Up_in_cancer', 'Down_in_cancer'
)
panel.to_csv(OUT / 'frozen_NIBFS_top20_used.csv', index=False)

cfg = rna.RNASeqValidationConfig(
    bootstrap_replicates=2000,
    random_state=42,
    download_workers=8,
    parse_workers=4,
    resume=True,
    force_rerun=False,
    smoke_test_pairs=None,
)
cfg.validate()
samples = rna._query_tcga_brca_samples(cfg)
samples.to_csv(OUT / 'TCGA_BRCA_candidate_paired_samples.csv', index=False)

def query_star_count_files_fixed(selected_samples, config):
    """
    Query all open TCGA-BRCA STAR-Counts files with the current GDC field
    namespace, then link them locally to the selected paired samples.
    """
    payload = {
        "filters": {
            "op": "and",
            "content": [
                {
                    "op": "in",
                    "content": {
                        "field": "cases.project.project_id",
                        "value": [config.project_id],
                    },
                },
                {
                    "op": "in",
                    "content": {
                        "field": "files.data_category",
                        "value": ["Transcriptome Profiling"],
                    },
                },
                {
                    "op": "in",
                    "content": {
                        "field": "files.data_type",
                        "value": ["Gene Expression Quantification"],
                    },
                },
                {
                    "op": "in",
                    "content": {
                        "field": "files.analysis.workflow_type",
                        "value": [config.workflow_type],
                    },
                },
                {
                    "op": "in",
                    "content": {
                        "field": "files.access",
                        "value": ["open"],
                    },
                },
            ],
        },
        "format": "JSON",
        "fields": ",".join(
            [
                "file_id",
                "file_name",
                "md5sum",
                "file_size",
                "created_datetime",
                "updated_datetime",
                "analysis.workflow_type",
                "cases.case_id",
                "cases.submitter_id",
                "cases.samples.sample_id",
                "cases.samples.submitter_id",
                "cases.samples.sample_type",
            ]
        ),
        "expand": "cases.samples",
        "sort": "updated_datetime:desc",
        "size": "5000",
    }

    response = rna._post_json(
        "files",
        payload,
        timeout=config.request_timeout_seconds,
        retries=config.request_retries,
    )

    hits = response.get("data", {}).get("hits", []) or []
    print("GDC STAR-Counts file hits:", len(hits))

    if not hits:
        raise RuntimeError(
            "GDC returned zero TCGA-BRCA STAR-Counts files."
        )

    selected = selected_samples.copy()
    selected["sample_id"] = selected["sample_id"].astype(str)
    selected["sample_submitter_id"] = (
        selected["sample_submitter_id"].astype(str)
    )
    selected["case_id"] = selected["case_id"].astype(str)

    selected_by_uuid = selected.set_index("sample_id", drop=False)
    selected_by_barcode = selected.set_index(
        "sample_submitter_id",
        drop=False,
    )

    rows = []

    for hit in hits:
        nested_samples = []

        for case in hit.get("cases", []) or []:
            case_id = str(case.get("case_id", ""))
            case_submitter = str(case.get("submitter_id", ""))

            for sample in case.get("samples", []) or []:
                nested_samples.append(
                    {
                        "case_id": case_id,
                        "case_submitter_id": case_submitter,
                        "sample_id": str(sample.get("sample_id", "")),
                        "sample_submitter_id": str(
                            sample.get("submitter_id", "")
                        ),
                        "sample_type": str(
                            sample.get("sample_type", "")
                        ),
                    }
                )

        matched_sample_ids = set()

        for nested in nested_samples:
            sample_uuid = nested["sample_id"]
            if sample_uuid in selected_by_uuid.index:
                matched_sample_ids.add(sample_uuid)

        if not matched_sample_ids:
            for nested in nested_samples:
                barcode = nested["sample_submitter_id"]
                if barcode in selected_by_barcode.index:
                    matched_sample_ids.add(
                        str(selected_by_barcode.loc[barcode, "sample_id"])
                    )

        if not matched_sample_ids:
            for nested in nested_samples:
                candidates = selected[
                    selected["case_id"].eq(nested["case_id"])
                    & selected["sample_type"].eq(nested["sample_type"])
                ]
                if len(candidates) == 1:
                    matched_sample_ids.add(
                        str(candidates.iloc[0]["sample_id"])
                    )

        for sample_id in sorted(matched_sample_ids):
            rows.append(
                {
                    "sample_id": sample_id,
                    "file_id": str(hit.get("file_id", "")),
                    "file_name": str(hit.get("file_name", "")),
                    "md5sum": str(hit.get("md5sum", "")),
                    "file_size": hit.get("file_size"),
                    "created_datetime": hit.get("created_datetime"),
                    "updated_datetime": hit.get("updated_datetime"),
                    "workflow_type": (
                        hit.get("analysis", {}) or {}
                    ).get("workflow_type"),
                }
            )

    files = pd.DataFrame(rows)

    if files.empty:
        debug_rows = []

        for hit in hits[:20]:
            for case in hit.get("cases", []) or []:
                for sample in case.get("samples", []) or []:
                    debug_rows.append(
                        {
                            "file_id": hit.get("file_id"),
                            "file_name": hit.get("file_name"),
                            "case_id": case.get("case_id"),
                            "case_submitter_id": case.get("submitter_id"),
                            "sample_id": sample.get("sample_id"),
                            "sample_submitter_id": sample.get("submitter_id"),
                            "sample_type": sample.get("sample_type"),
                        }
                    )

        debug_path = OUT / "GDC_STAR_Counts_linkage_debug.csv"
        pd.DataFrame(debug_rows).to_csv(debug_path, index=False)

        raise RuntimeError(
            "GDC returned STAR-Counts files, but none could be linked "
            "to the paired samples. Debug metadata saved to: "
            + str(debug_path)
        )

    files["updated_datetime_sort"] = pd.to_datetime(
        files["updated_datetime"],
        errors="coerce",
        utc=True,
    )

    files = (
        files.sort_values(
            ["sample_id", "updated_datetime_sort", "file_id"],
            ascending=[True, False, True],
        )
        .drop_duplicates("sample_id")
        .drop(columns=["updated_datetime_sort"])
    )

    merged = selected.merge(
        files,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )

    pair_complete = (
        merged.groupby("Pair_ID")["file_id"]
        .apply(lambda values: values.notna().sum() == 2)
    )
    complete_pair_ids = pair_complete[pair_complete].index

    merged = merged[
        merged["Pair_ID"].isin(complete_pair_ids)
        & merged["file_id"].notna()
    ].copy()

    if config.smoke_test_pairs is not None:
        keep_pairs = sorted(
            merged["Pair_ID"].unique()
        )[: int(config.smoke_test_pairs)]
        merged = merged[
            merged["Pair_ID"].isin(keep_pairs)
        ].copy()

    n_pairs = int(merged["Pair_ID"].nunique())
    required_pairs = (
        min(
            config.minimum_pairs,
            int(config.smoke_test_pairs),
        )
        if config.smoke_test_pairs is not None
        else config.minimum_pairs
    )

    print("Complete paired STAR-Counts cases:", n_pairs)

    if n_pairs < required_pairs:
        linkage_audit = selected.merge(
            files,
            on="sample_id",
            how="left",
        )
        audit_path = OUT / "GDC_STAR_Counts_pair_linkage_audit.csv"
        linkage_audit.to_csv(audit_path, index=False)

        raise RuntimeError(
            f"Only {n_pairs} complete paired cases were linked; "
            f"at least {required_pairs} were required. "
            f"Audit saved to {audit_path}"
        )

    pair_check = merged.groupby("Pair_ID")["Label"].agg(
        ["count", "nunique"]
    )

    if not (
        (pair_check["count"] == 2)
        & (pair_check["nunique"] == 2)
    ).all():
        raise RuntimeError(
            "Linked data do not contain exactly one tumor and one "
            "normal sample per participant."
        )

    return merged.sort_values(
        ["Pair_ID", "Label"]
    ).reset_index(drop=True)


manifest = query_star_count_files_fixed(samples, cfg)
manifest.to_csv(
    OUT / "TCGA_BRCA_selected_STAR_Counts_manifest.csv",
    index=False,
)
proc = OUT / 'TCGA_BRCA_paired_log2_TPM_full.csv.gz'
audit = OUT / 'TCGA_BRCA_download_audit.csv'
if proc.is_file() and audit.is_file():
    Xext = pd.read_csv(proc, index_col=0, compression='gzip')
else:
    downloaded = rna._download_files(
        manifest, Path('/content/TCGA_BRCA_RNA_CACHE/star_counts'), cfg
    )
    downloaded.to_csv(audit, index=False)
    Xext = rna._build_external_expression(downloaded, cfg)
    Xext.to_csv(proc, compression='gzip')
Xext.index = Xext.index.astype(str)
Xext.columns = Xext.columns.astype(str).str.strip().str.upper()
Xext = Xext.loc[:, ~Xext.columns.duplicated()]
meta = manifest[
    manifest['sample_id'].astype(str).isin(Xext.index)
].sort_values(['Pair_ID', 'Label']).reset_index(drop=True)
Xext = Xext.loc[meta['sample_id'].astype(str)]
shared = sorted(set(Xdev.columns) & set(Xext.columns))
missing_panel = sorted(set(genes) - set(shared))
if len(shared) < 5000:
    raise RuntimeError(f'Shared genes hanya {len(shared)}')
if missing_panel:
    raise RuntimeError('Panel tidak lengkap di TCGA: ' + ', '.join(missing_panel))
print('External pairs:', meta['Pair_ID'].nunique(), '| shared genes:', len(shared), '| panel: 20/20')

def rank_rows(df):
    out = df.rank(axis=1, method='average', pct=True)
    if out.isna().any().any():
        raise RuntimeError('Rank menghasilkan NA')
    return out

D = rank_rows(Xdev[shared])[genes]
E = rank_rows(Xext[shared])[genes]
D.to_csv(OUT / 'development_rank_features_frozen_top20.csv.gz', compression='gzip')
E.to_csv(OUT / 'TCGA_BRCA_rank_features_frozen_top20.csv.gz', compression='gzip')

from lightgbm import LGBMClassifier
models = {
    'LR': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(
            C=1.0, penalty='l2', solver='lbfgs', max_iter=5000, random_state=42
        )),
    ]),
    'RF': RandomForestClassifier(
        n_estimators=500, max_features='sqrt', class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        objective='binary', n_estimators=500, learning_rate=0.03,
        num_leaves=31, subsample=0.90, colsample_bytree=0.90,
        class_weight='balanced', random_state=42, n_jobs=-1, verbosity=-1
    ),
}

rows = []
signs = panel.set_index('Gene')['Discovery_logFC'].map(np.sign)
signed = E.mul(signs, axis=1).mean(axis=1)
for sid, score in signed.items():
    row = meta.loc[meta['sample_id'].astype(str).eq(str(sid))].iloc[0]
    rows.append({
        'sample_id': sid, 'Pair_ID': row.Pair_ID, 'case_id': row.case_id,
        'case_submitter_id': row.case_submitter_id, 'sample_type': row.sample_type,
        'Label': int(row.Label), 'Model': 'FrozenSignedPanelScore',
        'Score': float(score), 'Threshold': 0.0,
        'Representation': 'within-sample percentile rank',
        'External_labels_used_for_fitting': False,
    })
for name, estimator in models.items():
    fitted = clone(estimator).fit(D, ydev.loc[D.index])
    probabilities = fitted.predict_proba(E)[:, 1]
    for sid, score in zip(E.index, probabilities):
        row = meta.loc[meta['sample_id'].astype(str).eq(str(sid))].iloc[0]
        rows.append({
            'sample_id': sid, 'Pair_ID': row.Pair_ID, 'case_id': row.case_id,
            'case_submitter_id': row.case_submitter_id, 'sample_type': row.sample_type,
            'Label': int(row.Label), 'Model': name, 'Score': float(score),
            'Threshold': 0.5, 'Representation': 'within-sample percentile rank',
            'External_labels_used_for_fitting': False,
        })
pred = pd.DataFrame(rows)
pred.to_csv(OUT / 'TCGA_BRCA_RNAseq_predictions.csv', index=False)

metrics, boots = rna._pair_bootstrap_metrics(
    pred, bootstrap_replicates=2000, random_state=42
)
for name, block in pred.groupby('Model', sort=False):
    metrics.loc[len(metrics)] = {
        'Model': name, 'Metric': 'Brier',
        'Estimate': brier_score_loss(block.Label, np.clip(block.Score, 0, 1)),
        'CI95_low': np.nan, 'CI95_high': np.nan,
        'Bootstrap_unit': 'TCGA participant pair',
        'Bootstrap_replicates_completed': 0,
        'Threshold': float(block.Threshold.iloc[0]),
    }
metrics.to_csv(
    OUT / 'TCGA_BRCA_RNAseq_performance_with_pair_bootstrap_CI.csv', index=False
)
boots.to_csv(
    OUT / 'TCGA_BRCA_RNAseq_pair_bootstrap_distribution.csv.gz',
    index=False, compression='gzip'
)
rep = rna._paired_gene_replication(Xext[genes], meta, panel)
rep.to_csv(OUT / 'TCGA_BRCA_gene_direction_replication.csv', index=False)
rna._plot_roc(pred, OUT)
rna._plot_direction_replication(rep, OUT)

wide = metrics.pivot(
    index='Model', columns='Metric', values=['Estimate', 'CI95_low', 'CI95_high']
)
wide.columns = [f'{a}_{b}' for a, b in wide.columns]
wide = wide.reset_index()
wide.insert(1, 'External_cohort', 'TCGA-BRCA paired RNA-seq')
wide.insert(2, 'Pairs', int(meta.Pair_ID.nunique()))
wide.to_csv(
    OUT / 'Supplementary_Table_SXX_TCGA_BRCA_RNAseq_validation.csv', index=False
)
rep.to_csv(
    OUT / 'Supplementary_Table_SXY_TCGA_BRCA_gene_replication.csv', index=False
)

pd.DataFrame([{
    'Archive': str(project_archive),
    'Harmonized_matrix': str(MATRIX),
    'Split_assignments': str(SPLIT),
    'Frozen_panel': str(PANEL),
    'Development_samples': len(Xdev),
    'Development_cancer': int(ydev.sum()),
    'Development_normal': int((1-ydev).sum()),
    'Shared_genes': len(shared),
    'Panel_coverage': '20/20',
    'External_labels_used_for_fitting': False,
    'Old_notebook_rerun': False,
}]).to_csv(OUT / 'TCGA_BRCA_validation_source_audit.csv', index=False)

summary = {
    'analysis': 'TCGA_BRCA_RNAseq_from_archived_harmonized_matrix',
    'development_samples': len(Xdev),
    'external_pairs': int(meta.Pair_ID.nunique()),
    'external_samples': len(meta),
    'shared_gene_universe': len(shared),
    'complete_panel_coverage': True,
    'direction_concordant_genes': int(rep.Direction_concordant.sum()),
    'significant_genes_FDR_0.05': int(rep['Significant_FDR_0.05'].sum()),
    'bootstrap_replicates': 2000,
    'external_labels_used_for_fitting': False,
    'old_notebook_rerun': False,
}
(OUT / 'TCGA_BRCA_RNAseq_validation_summary.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8'
)

auc = metrics[metrics.Metric.eq('ROC_AUC')]
auc_text = '; '.join(
    f'{row.Model}: {row.Estimate:.4f} '
    f'(95% CI {row.CI95_low:.4f}–{row.CI95_high:.4f})'
    for row in auc.itertuples(index=False)
)
manuscript_text = (
    'SUPPLEMENTARY METHODS\n\n'
    'Independent TCGA-BRCA RNA-seq validation used the exact frozen NIBFS '
    'top-20 panel. Transfer models were fitted only on the 608 archived '
    'model-development samples. Microarray and RNA-seq values were separately '
    f'converted to within-sample percentile ranks over {len(shared)} shared genes. '
    'External labels were used only for evaluation. Confidence intervals used '
    '2,000 participant-pair bootstrap resamples.\n\n'
    'SUPPLEMENTARY RESULTS\n\n'
    f'The analysis included {meta.Pair_ID.nunique()} participant pairs with '
    f'complete 20/20 panel coverage. ROC-AUC: {auc_text}. Discovery directions '
    f'replicated for {int(rep.Direction_concordant.sum())}/20 genes; '
    f'{int(rep['Significant_FDR_0.05'].sum())}/20 were significant at BH-FDR < 0.05.\n\n'
    'INTERPRETATION\n\n'
    'This is frozen-panel cross-technology validation, not application of an '
    'unchanged microarray-scale fitted model.\n'
)
(OUT / 'manuscript_text_TCGA_BRCA_RNAseq_validation.txt').write_text(
    manuscript_text, encoding='utf-8'
)

result_zip = Path(shutil.make_archive(
    str(OUT), 'zip', root_dir=OUT.parent, base_dir=OUT.name
))
backup_zip = BACKUP / 'TCGA_BRCA_RNASEQ_EXTERNAL_VALIDATION.zip'
shutil.copy2(result_zip, backup_zip)
print('\n' + '=' * 76)
print('TCGA-BRCA RNA-SEQ VALIDATION COMPLETE')
print('=' * 76)
print('External pairs:', summary['external_pairs'])
print('Shared genes:', summary['shared_gene_universe'])
print('Top-20 coverage:', summary['complete_panel_coverage'])
print('Results:', OUT)
print('Backup ZIP:', backup_zip)
